# Joins and Aggregation

Relational questions combine tables and summarise them. This tutorial builds customers, products and orders, joins them, aggregates them with `GROUP BY` and window functions, and checks one result in Julia.

It needs DuckDB.jl, DBInterface.jl and Tables.jl, and uses an **in-memory** database.

In [ ]:
# DuckDB is built into Pollis: use its environment if this project does not have DuckDB
pollis_env = joinpath(homedir(), ".pollis", "julia", "environments", "v$(VERSION.major).$(VERSION.minor)")
isdir(pollis_env) && !(pollis_env in LOAD_PATH) && push!(LOAD_PATH, pollis_env)

using DuckDB, DBInterface, Tables, Statistics

con = DBInterface.connect(DuckDB.DB, ":memory:")
query(sql) = Tables.columntable(DBInterface.execute(con, sql))

## Three Tables

Every order points to a customer and a product. Order 13 points to customer 99, who does not exist, as happens in real data.

In [ ]:
DBInterface.execute(con, """
CREATE TABLE customers AS
SELECT i AS customer_id, 'C' || i AS name, ['north', 'south', 'east'][1 + i % 3] AS region
FROM range(1, 21) t(i)
""")
DBInterface.execute(con, """
CREATE TABLE products AS
SELECT * FROM (VALUES (1, 'tea', 4.5), (2, 'coffee', 7.0), (3, 'cocoa', 5.25)) p(product_id, product, unit_price)
""")
DBInterface.execute(con, """
CREATE TABLE orders AS
SELECT i AS order_id,
       CASE WHEN i = 13 THEN 99 ELSE 1 + i * 7 % 20 END AS customer_id,
       1 + i % 3 AS product_id,
       1 + i * 5 % 9 AS qty,
       DATE '2026-01-01' + (i * 3)::INTEGER AS day
FROM range(1, 121) t(i)
""")
query("SELECT table_name, estimated_size FROM duckdb_tables() ORDER BY table_name")

## Inner and Left Joins

An inner `JOIN` keeps only matching rows, so order 13 disappears. A `LEFT JOIN` keeps every order and fills the missing customer with `NULL`, which Julia sees as `missing`.

In [ ]:
inner = query("SELECT count(*) AS n FROM orders JOIN customers USING (customer_id)")
left  = query("SELECT count(*) AS n, count(name) AS matched FROM orders LEFT JOIN customers USING (customer_id)")
inner, left

An **anti join** finds the orders without a customer directly:

In [ ]:
query("SELECT * FROM orders ANTI JOIN customers USING (customer_id)")

## Duplicate Keys Multiply Rows

If the right-hand table has a key twice, every matching row on the left is repeated. Always check that join keys are unique.

In [ ]:
DBInterface.execute(con, "CREATE TABLE products_dup AS SELECT * FROM products UNION ALL SELECT 2, 'coffee (new price)', 7.5")
query("SELECT (SELECT count(*) FROM orders) AS orders, count(*) AS joined FROM orders JOIN products_dup USING (product_id)")

In [ ]:
query("SELECT product_id, count(*) AS n FROM products_dup GROUP BY product_id HAVING count(*) > 1")

## Group and Aggregate

Revenue by region and product. `GROUP BY ALL` groups by every column that is not aggregated; `HAVING` filters the groups.

In [ ]:
revenue = query("""
SELECT c.region, p.product, count(*) AS orders, sum(o.qty * p.unit_price) AS revenue
FROM orders o
JOIN customers c USING (customer_id)
JOIN products p USING (product_id)
GROUP BY ALL
HAVING count(*) >= 10
ORDER BY c.region, revenue DESC
""")

## Window Functions

A window function computes a value for each row from a group of rows, without collapsing them. Here, each customer's running total and rank within their region.

In [ ]:
query("""
WITH totals AS (
	SELECT c.region, c.name, sum(o.qty * p.unit_price) AS spent
	FROM orders o JOIN customers c USING (customer_id) JOIN products p USING (product_id)
	GROUP BY ALL
)
SELECT region, name, spent,
       rank() OVER (PARTITION BY region ORDER BY spent DESC) AS rank_in_region,
       round(100 * spent / sum(spent) OVER (PARTITION BY region), 1) AS pct_of_region
FROM totals
QUALIFY rank_in_region <= 2
ORDER BY region, rank_in_region
""")

A moving average over the previous seven orders, in date order:

In [ ]:
query("""
SELECT day, qty,
       round(avg(qty) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS moving_avg
FROM orders
ORDER BY day
LIMIT 10
""")

## Check in Julia

Fetch the joined rows and compute one revenue figure in Julia. The unit prices are SQL `DECIMAL` values, which arrive as exact `FixedDecimal` numbers, so the two totals match to the cent rather than only approximately.

In [ ]:
rows = query("""
SELECT c.region, p.product, o.qty * p.unit_price AS amount
FROM orders o JOIN customers c USING (customer_id) JOIN products p USING (product_id)
""")
pick = (rows.region .== "north") .& (rows.product .== "tea")
julia_total = sum(rows.amount[pick])
sql_total = revenue.revenue[findfirst(i -> revenue.region[i] == "north" && revenue.product[i] == "tea", eachindex(revenue.region))]
julia_total, sql_total, julia_total == sql_total

## How DuckDB Ran It

`EXPLAIN ANALYZE` shows the plan: read it from the bottom up, from the table scans to the final aggregate, with the rows each step produced.

In [ ]:
plan = query("EXPLAIN ANALYZE SELECT c.region, sum(o.qty) FROM orders o JOIN customers c USING (customer_id) GROUP BY ALL")
println(last(plan)[1])

In [ ]:
DBInterface.close!(con)